# 03 — Ablations

Each cell here answers one question with a number instead of an argument. Every
choice the pipeline makes — window length, detrending method, skin masking,
Welch vs a single FFT — is defended or overturned by a measurement.

**These are synthetic-ground-truth results.** They establish that a design
choice helps *the DSP*, under noise and distortion we control exactly. They do
not establish real-world magnitudes; that needs UBFC-rPPG, which is notebook
02. Read every number here as "this is the right choice, and here is why",
never as "this is the accuracy you will get on a face".


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rppg.synthetic import synth_rgb, nonuniform_timestamps
from rppg.pipeline import PipelineConfig, analyse_signal
from rppg.metrics import mae, rmse, pearson
from rppg.spectral import bpm_resolution
from rppg.methods import PROJECTIONS

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9,
                     "figure.facecolor": "white"})
COL = {"green": "#2e9e4f", "ica": "#c2410c", "chrom": "#1d4ed8", "pos": "#7c3aed"}
FS = 30.0
N_SUBJECTS = 4          # independent clips per cell; guards against one-clip flukes
DURATION = 70.0
METHODS = list(PROJECTIONS)


## The harness

One function used by every ablation below: generate `N_SUBJECTS` independent
clips, run the pipeline, and score every window against the ground truth that
was injected. Each subject gets a different resting HR, so no result can come
from a method that happens to sit near one particular value.


In [ ]:
def run(cfg=None, n_subjects=N_SUBJECTS, duration=DURATION, bpm=None, **synth_kw):
    """Score the pipeline over several synthetic subjects. Returns per-window rows."""
    cfg = cfg or PipelineConfig(fs=FS)
    frames = []
    for s in range(n_subjects):
        rng = 1000 + s
        # A different resting HR per subject, spread across the plausible range.
        subject_bpm = (bpm(s) if callable(bpm)
                       else bpm if bpm is not None else float(58 + 9 * s))
        ts = nonuniform_timestamps(duration, FS, jitter_ms=3.0, drop_prob=0.003, rng=rng)
        clip = synth_rgb(timestamps=ts, bpm=subject_bpm, rng=rng, **synth_kw)
        df = analyse_signal(clip.t, clip.rgb, cfg)
        df["bpm_ref"] = np.interp(df.t_center, clip.t, clip.bpm_true)
        df["subject"] = s
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def score(df, col="bpm_smoothed", accepted_only=True):
    """Per-method MAE / RMSE / r / accept-rate."""
    rows = []
    for m in sorted(df.method.unique()):
        sub = df[df.method == m]
        used = sub[sub.accepted] if accepted_only else sub
        rows.append(dict(
            method=m,
            MAE=mae(used[col], used.bpm_ref),
            RMSE=rmse(used[col], used.bpm_ref),
            r=pearson(used[col], used.bpm_ref),
            accept=sub.accepted.mean(),
            n=len(used),
        ))
    return pd.DataFrame(rows).set_index("method")


baseline = run()
print("baseline: 15 s window, smoothness detrend, single FFT, 8x zero-pad")
print(score(baseline).round(3).to_string())


## 1. Window length — the central trade-off

Frequency resolution is $60/T$ BPM, so a longer window resolves better. But a
longer window also averages over more of a *changing* heart rate, and gives
motion more opportunity to contaminate it.

Two choices decide whether this sweep measures anything:

- **The HR profile must be curved.** Against a *linear* ramp the window costs
  nothing at all: the mean HR over a window equals the instantaneous HR at its
  centre, so the lag cancels exactly and the sweep reports "longer is always
  better" no matter how long the window gets. A step change is the honest test.
- **Score the raw per-window estimate, not the median-filtered one.** The
  5-deep median has a lag of its own, and it swamps the effect being measured —
  it is a property of the smoother, not of the window.

Both the constant and stepped cases are below; the difference between them is
the result.


In [ ]:
WINDOWS = [5, 8, 10, 12, 15, 20, 25, 30]

# HR holds at 70, steps to 105 half way through, and holds. A long window
# straddling the step averages two different heart rates and can report neither.
def stepped(subject):
    # A different step position per subject. With one fixed step time, the MAE
    # depends on exactly where window centres happen to land relative to it,
    # which aliases against T and puts spurious kinks in the curve.
    k = int(100 * (0.35 + 0.10 * subject))
    return np.concatenate([np.full(k, 70.0), np.full(100 - k, 105.0)])

rows = []
for T in WINDOWS:
    print(f"  window {T}s...", flush=True)
    cfg = PipelineConfig(fs=FS, window_sec=float(T), hop_sec=3.0)
    for label, kw in [("constant HR", {}), ("stepped HR", dict(bpm=stepped))]:
        df = run(cfg, duration=120.0, **kw)
        # Raw per-window estimate: the median filter's own lag would otherwise
        # dominate and flatten the curve we are trying to see.
        s = score(df, col="bpm")
        for m in METHODS:
            rows.append(dict(T=T, case=label, method=m,
                             MAE=s.loc[m, "MAE"], accept=s.loc[m, "accept"]))
sweep = pd.DataFrame(rows)
print(sweep.pivot_table(index="T", columns=["case", "method"], values="MAE").round(2).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, case in zip(axes, ["constant HR", "stepped HR"]):
    sub = sweep[sweep.case == case]
    for m in METHODS:
        s = sub[sub.method == m]
        ax.plot(s["T"], s.MAE, marker="o", ms=4, lw=1.5, color=COL[m], label=m.upper())
    ax.plot(WINDOWS, [bpm_resolution(t) / 2 for t in WINDOWS], "k:", lw=1.2,
            label="½ × 60/T (quantisation floor)")
    ax.set(xlabel="window length T (s)", title=case)
    ax.set_yscale("log")
axes[0].set_ylabel("MAE (BPM)")
axes[0].legend(fontsize=7, frameon=False, ncol=2)
fig.suptitle("MAE vs window length — the trade-off only appears when HR changes abruptly", y=1.02)
fig.tight_layout()

best = (sweep[sweep.case == "stepped HR"]
        .loc[lambda d: d.groupby("method").MAE.idxmin(), ["method", "T", "MAE"]])
print("best T per method, stepped HR:")
print(best.to_string(index=False))


### Reading it

Against a constant HR, MAE falls monotonically with $T$ and tracks the $60/T$
quantisation floor — exactly as theory says, and exactly why that sweep alone
would mislead you into picking the longest window available.

Against a stepped HR the curve turns: past some $T$ the window straddles the
step, averages two different heart rates, and reports neither. The error from
*lag* overtakes the error from *resolution*. The minimum of that curve is the
defensible choice, and it is the plot to put in the report.

Worth stating explicitly in the viva, because it is counter-intuitive: against
a **linear** ramp there is no penalty at any window length, since the mean HR
over a window equals the instantaneous HR at its centre. Window length only
costs you where the HR profile is curved.


## 2. Detrending

Respiration puts a large 0.2–0.4 Hz baseline under the cardiac signal.
Smoothness-priors detrending (Tarvainen et al.) is a time-varying high-pass
with a very smooth response; a moving-average subtraction is the cheap
alternative. Does the sophistication pay?

The test is run at a **high respiration amplitude**, because that is where the
two can actually differ — at low respiration both work and the comparison says
nothing.


In [ ]:
rows = []
for method_name in ["smoothness", "moving_average", "none"]:
    print(f"  detrend {method_name}...", flush=True)
    for resp_amp in [0.02, 0.06, 0.12]:
        cfg = PipelineConfig(fs=FS, detrend_method=method_name)
        df = run(cfg, respiration_amplitude=resp_amp)
        s = score(df)
        for m in METHODS:
            rows.append(dict(detrend=method_name, resp=resp_amp, method=m,
                             MAE=s.loc[m, "MAE"], SNR=df[df.method == m].snr.mean()))
detr = pd.DataFrame(rows)
print(detr.pivot_table(index=["resp", "detrend"], columns="method", values="MAE").round(2).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
piv = detr.pivot_table(index="resp", columns="detrend", values="MAE")
piv.plot(kind="bar", ax=axes[0], color=["#94a3b8", "#ea580c", "#1d4ed8"], width=0.75)
axes[0].set(xlabel="respiration amplitude", ylabel="MAE (BPM), mean over arms",
            title="detrending vs respiration strength")
axes[0].legend(fontsize=8, frameon=False, title=None)
axes[0].tick_params(axis="x", rotation=0)

piv_snr = detr.pivot_table(index="resp", columns="detrend", values="SNR")
piv_snr.plot(kind="bar", ax=axes[1], color=["#94a3b8", "#ea580c", "#1d4ed8"], width=0.75)
axes[1].set(xlabel="respiration amplitude", ylabel="mean SNR (dB)", title="…and on SNR")
axes[1].legend(fontsize=8, frameon=False)
axes[1].tick_params(axis="x", rotation=0)
fig.tight_layout()


## 3. Motion robustness — the project's central claim, as a curve

The claim is that model-based projections (CHROM, POS) survive motion that
breaks the non-model-based ones (GREEN, ICA), because motion enters through a
shared multiplicative illumination term rather than as an independent additive
source.

Sweeping the motion amplitude turns that claim into a measurement with a
crossing point.


In [ ]:
AMPS = [0.0, 0.005, 0.01, 0.02, 0.035, 0.05, 0.08]
rows = []
for amp in AMPS:
    print(f"  motion {amp}...", flush=True)
    df = run(PipelineConfig(fs=FS), motion_amplitude=amp, motion_bpm=42.0)
    s = score(df)
    for m in METHODS:
        rows.append(dict(amp=amp, method=m, MAE=s.loc[m, "MAE"], SNR=df[df.method == m].snr.mean()))
motion = pd.DataFrame(rows)
print(motion.pivot_table(index="amp", columns="method", values="MAE").round(2).to_string())


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for m in METHODS:
    s = motion[motion.method == m]
    ax[0].plot(np.array(AMPS) * 100, s.MAE, marker="o", ms=4, lw=1.6, color=COL[m], label=m.upper())
    ax[1].plot(np.array(AMPS) * 100, s.SNR, marker="o", ms=4, lw=1.6, color=COL[m])
ax[0].set(xlabel="specular motion amplitude (% of DC)", ylabel="MAE (BPM)",
          title="model-based arms hold; GREEN and ICA do not")
ax[0].legend(fontsize=8, frameon=False)
ax[1].set(xlabel="specular motion amplitude (% of DC)", ylabel="mean SNR (dB)",
          title="SNR rises for the arms that are WRONG")
fig.tight_layout()

worst = motion[motion.amp == AMPS[-1]].set_index("method")
print("at the strongest motion tested:")
for m in METHODS:
    print(f"  {m:>6s}  MAE {worst.loc[m,'MAE']:7.2f} BPM   mean SNR {worst.loc[m,'SNR']:+6.2f} dB")


### The finding that matters most in this notebook

Look at the right-hand panel. As motion grows, the **SNR of GREEN and ICA
rises** — while their MAE rises with it. They lock onto the motion frequency,
which is a genuinely narrowband signal, and report high confidence in a wrong
answer.

**SNR measures narrowbandness, not correctness.** The quality metric cannot
detect this failure mode, by construction. That is the argument for reporting
cross-method agreement alongside SNR, and ultimately for validating against
ground truth rather than against a self-assessment.


## 4. Skin masking

The YCrCb mask drops non-skin pixels inside the ROI. It costs a colour-space
conversion and two morphological operations per frame. Worth it?

This needs pixels, not signals, so it runs on a rendered video: a skin-coloured
disc on a dark background, framed two ways — an ROI **tight** inside the skin,
and one that **overhangs** so background leaks in, which is the situation a
landmark box is actually in when it catches hair, eyebrows or the jaw edge.

Two details decide whether this experiment measures anything at all:

- The noise has to be **per-pixel**. Spatial averaging reduces noise as
  $1/\sqrt{N}$, so masking helps by raising the *fraction* of averaged pixels
  that carry pulse. Noise common to every pixel — the kind baked into a
  synthetic signal before rendering — is untouched by averaging, so an ablation
  run under it measures nothing and will report that masking is useless.
- The ROI has to actually overhang. If it is already inside the skin, masking
  can only remove pixels, and removing pixels costs SNR.

Getting either wrong produces a confident null result. Both cases are run below
so the difference is visible rather than assumed.


In [ ]:
import cv2
from rppg.roi import spatial_mean, _rect_poly

W, H, RADIUS = 320, 240, 70
CENTRE = (W // 2, H // 2)


def render_and_average(clip, rect, use_mask, pixel_noise=2.0, rng=0):
    """Synthesise frames in memory and take the ROI mean. No codec involved.

    Deliberately not routed through a video file: every codec available here is
    lossy enough to swamp the effect being measured (`mp4v` misreads this
    material by ~50 levels against a ~2-level pulse), and even a lossless codec
    only adds I/O to an experiment that needs none.
    """
    rng = np.random.default_rng(rng)
    polys = [_rect_poly(*rect)]
    # Composite in float and quantise once, at the end. Drawing into a uint8
    # array would round the disc colour before the noise is added, and a
    # sub-level pulse cannot survive that: dither only linearises a quantiser it
    # precedes.
    alpha_u8 = np.zeros((H, W), np.uint8)
    cv2.circle(alpha_u8, CENTRE, RADIUS, 255, -1, lineType=cv2.LINE_AA)
    alpha = (alpha_u8.astype(float) / 255.0)[:, :, None]

    out, npx = [], []
    for row in clip.rgb:
        colour = np.asarray(row[::-1], dtype=float)
        frame = 40.0 + (colour[None, None, :] - 40.0) * alpha
        frame = frame + rng.normal(0, pixel_noise, size=(H, W, 3))
        rgb, n, _ = spatial_mean(np.clip(np.round(frame), 0, 255).astype(np.uint8),
                                 polys, use_mask)
        out.append(rgb)
        npx.append(n)
    return np.array(out), float(np.mean(npx))


FRAMINGS = {
    "tight (inside the skin)": (int(0.36 * W), int(0.34 * H), int(0.28 * W), int(0.32 * H)),
    "loose (overhangs)": (int(0.10 * W), int(0.05 * H), int(0.80 * W), int(0.90 * H)),
}

rows = []
for s in range(3):
    # noise_std near zero: the noise that matters here must be per-pixel, so it
    # is injected at render time instead, where spatial averaging can act on it.
    clip = synth_rgb(duration=45.0, fs=FS, bpm=float(62 + 8 * s),
                     pulse_amplitude=0.02, noise_std=0.005, rng=200 + s)
    for framing, rect in FRAMINGS.items():
        for use_mask in (True, False):
            rgb, px = render_and_average(clip, rect, use_mask, rng=200 + s)
            df = analyse_signal(clip.t, rgb,
                                PipelineConfig(fs=FS, window_sec=15.0, hop_sec=3.0))
            df["bpm_ref"] = np.interp(df.t_center, clip.t, clip.bpm_true)
            sc = score(df)
            for m in METHODS:
                rows.append(dict(framing=framing, mask=use_mask, subject=s, method=m,
                                 px=px, MAE=sc.loc[m, "MAE"],
                                 SNR=df[df.method == m].snr.mean()))
mask_df = pd.DataFrame(rows)

print("mean ROI pixels entering the average:")
print(mask_df.groupby(["framing", "mask"]).px.mean().round(0).to_string())
print()
print("SNR (dB), mean over arms:")
print(mask_df.pivot_table(index="framing", columns="mask", values="SNR").round(2).to_string())
print()
for framing in FRAMINGS:
    sub = mask_df[mask_df.framing == framing]
    on = sub[sub["mask"]].SNR.mean()
    off = sub[~sub["mask"]].SNR.mean()
    print(f"{framing:26s} masking: {off:+6.2f} -> {on:+6.2f} dB  ({on - off:+.2f} dB)")


## 4b. How a sub-level pulse survives 8-bit quantisation at all

A 0.4% modulation on mid-grey skin is about **0.74 levels peak-to-peak** —
smaller than one 8-bit step. It ought to be unrepresentable. It is recovered
anyway, and the mechanism is worth understanding because two easy mistakes
destroy it silently.

**Dither.** Per-pixel sensor noise randomises the rounding error independently
across pixels, so the spatial average resolves below one LSB. Without it, every
pixel of a uniform patch rounds the same way, the mean of N identical values is
that value, and no amount of averaging recovers anything.

**Order matters.** Dither must precede the quantiser. Noise added after
rounding cannot restore information rounding already discarded.

**Re-quantisation.** A lossy codec quantises again, in a transform domain, with
bits allocated to what the eye notices — and a sub-percent uniform brightness
shift is exactly what it discards.


In [ ]:
import tempfile, os
from rppg.synthetic import write_synthetic_video
from rppg.capture import VideoFileSource
from rppg.roi import make_roi, extract_signal

tmpdir = tempfile.mkdtemp()
clip = synth_rgb(duration=30.0, fs=FS, bpm=72.0, pulse_amplitude=0.004,
                 noise_std=0.0, respiration_amplitude=0.0, rng=3)
truth = clip.rgb[:, 1] - clip.rgb[:, 1].mean()
print(f"injected pulse: {np.ptp(truth):.4f} levels peak-to-peak "
      f"(one 8-bit level = 1.0)")
print()

rows = []
for pixel_noise in (0.0, 2.0):
    for codec, ext in [("FFV1", ".avi"), ("mp4v", ".mp4")]:
        path = os.path.join(tmpdir, f"c_{pixel_noise}_{codec}{ext}")
        write_synthetic_video(path, clip, size=(W, H), radius=RADIUS,
                              pixel_noise=pixel_noise, codec=codec, rng=3)
        roi = make_roi("fixed", rel_rect=(0.36, 0.34, 0.28, 0.32), use_skin_mask=False)
        with VideoFileSource(path) as src:
            t, rgb, _ = extract_signal(src, roi)
        got = rgb[:, 1] - rgb[:, 1].mean()
        n = min(len(got), len(truth))
        r = (np.corrcoef(got[:n], truth[:n])[0, 1] if got[:n].std() > 1e-12 else np.nan)
        rows.append(dict(dither=pixel_noise, codec=codec, recovered_p2p=np.ptp(got[:n]),
                         corr=r, size_kb=os.path.getsize(path) / 1024))
comp = pd.DataFrame(rows)
print(comp.round(4).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
labels = [f"dither {r.dither:.0f} · {r.codec}" for r in comp.itertuples()]
vals = comp["corr"].fillna(0.0).values
ax.bar(labels, vals, color=["#dc2626" if v < 0.5 else "#16a34a" for v in vals])
ax.axhline(0.95, color="k", ls=":", lw=1.2, label="usable")
ax.set(ylabel="correlation with the injected pulse", ylim=(-0.2, 1.05),
       title="a sub-LSB pulse needs dither before the quantiser, and no re-quantisation")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()

for r in comp.itertuples():
    verdict = ("recovered" if r.corr > 0.95 else
               "degraded" if r.corr > 0.5 else "lost")
    print(f"dither {r.dither:.0f}, {r.codec:5s}: r = {r.corr:+.4f}  -> {verdict}")


### Consequences for the project

- **Never record the stress set to a lossy codec.** Compressed MP4 makes the
  benchmark measure the encoder, with no error message and a pipeline that
  looks perfectly healthy. UBFC-rPPG ships uncompressed 8-bit RGB for this
  reason. Record raw frames, or a lossless codec (FFV1).
- **Prefer `--save-signal`** — writing the RGB means straight out of the live
  capture avoids the question entirely.
- **A noiseless camera would be worse than a noisy one here**, which is a good
  viva answer: the sensor noise that dithers the quantiser is what makes a
  sub-LSB measurement possible in the first place.


## 5. Welch vs a single zero-padded FFT

Welch averages periodograms over overlapping sub-segments: lower variance,
worse resolution. A single FFT over the whole window keeps the resolution and
accepts the variance. Which wins depends on how noisy the signal is, so the
comparison is run across a range of noise levels.


In [ ]:
rows = []
for noise in [0.1, 0.3, 0.8, 1.5]:
    print(f"  noise {noise}...", flush=True)
    for spec in ["fft", "welch"]:
        cfg = PipelineConfig(fs=FS, spectrum=spec, window_sec=15.0)
        df = run(cfg, noise_std=noise)
        s = score(df)
        for m in METHODS:
            rows.append(dict(noise=noise, spectrum=spec, method=m, MAE=s.loc[m, "MAE"]))
spec_df = pd.DataFrame(rows)
piv = spec_df.pivot_table(index="noise", columns="spectrum", values="MAE")
piv["winner"] = np.where(piv.fft <= piv.welch, "fft", "welch")
print(piv.round(3).to_string())

fig, ax = plt.subplots(figsize=(5.5, 3.2))
for spec, c in [("fft", "#1d4ed8"), ("welch", "#ea580c")]:
    s = spec_df[spec_df.spectrum == spec].groupby("noise").MAE.mean()
    ax.plot(s.index, s.values, marker="o", ms=4, lw=1.6, color=c, label=spec)
ax.set(xlabel="sensor noise std (8-bit levels)", ylabel="MAE (BPM), mean over arms",
       title="single FFT vs Welch", xscale="log")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()


## 6. Zero-padding and peak interpolation

Zero-padding does not add resolution — but combined with parabolic
interpolation it locates an isolated peak far more precisely than the raw bin.
This measures how much of the accuracy comes from each.


In [ ]:
from rppg.spectral import estimate_bpm
from rppg.pipeline import project_and_filter
from rppg.resample import resample_uniform

# Scored at the spectrum, not through analyse_signal, because analyse_signal
# always interpolates — the "off" case has to be measured directly or the
# comparison is vacuous.
WIN_SEC = 10.0                      # 6 BPM raw bin spacing
cfg10 = PipelineConfig(fs=FS, window_sec=WIN_SEC)
rows = []
for s in range(3):
    ts = nonuniform_timestamps(70.0, FS, jitter_ms=3.0, rng=300 + s)
    clip = synth_rgb(timestamps=ts, bpm=float(58 + 9 * s) + 1.7, rng=300 + s)
    t_u, rgb_u = resample_uniform(clip.t, clip.rgb, fs=FS)
    win = int(WIN_SEC * FS)
    for start in range(0, len(rgb_u) - win + 1, int(2 * FS)):
        seg = rgb_u[start : start + win]
        ref = float(np.interp(t_u[start + win // 2], clip.t, clip.bpm_true))
        for m in METHODS:
            sig = project_and_filter(seg, FS, m, cfg10)
            for pad in [1, 2, 4, 8, 16]:
                for interp in (True, False):
                    r = estimate_bpm(sig, FS, zero_pad=pad, interpolate=interp)
                    rows.append(dict(pad=pad, interp=interp, method=m,
                                     err=abs(r.bpm - ref)))
pad_df = (pd.DataFrame(rows)
          .pivot_table(index="pad", columns="interp", values="err")
          .rename(columns={True: "parabolic", False: "raw bin"}))
print(f"MAE (BPM) at a {WIN_SEC:.0f} s window — raw bin spacing {bpm_resolution(WIN_SEC):.1f} BPM:")
print(pad_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(5.5, 3.2))
for col, c in [("raw bin", "#dc2626"), ("parabolic", "#16a34a")]:
    ax.plot(pad_df.index, pad_df[col], marker="o", ms=4, lw=1.6, color=c, label=col)
ax.axhline(bpm_resolution(WIN_SEC), color="k", ls=":", lw=1.2, label="bin spacing 60/T")
ax.set(xscale="log", xlabel="zero-padding factor", ylabel="MAE (BPM)",
       title="padding samples the DTFT; interpolation finds the vertex")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()

print()
print(f"raw bin, no padding   : {pad_df.loc[1, 'raw bin']:.2f} BPM")
print(f"8x padding, raw bin   : {pad_df.loc[8, 'raw bin']:.2f} BPM")
print(f"8x padding + parabolic: {pad_df.loc[8, 'parabolic']:.2f} BPM")
print("Both are well under the 6 BPM bin spacing — sub-bin accuracy from a")
print("spectrum whose actual resolution never changed.")


## Summary — what these ablations decided

| Question | Answer | Evidence |
|---|---|---|
| Window length | the minimum of the *ramping-HR* curve, not the longest available | §1 |
| Detrending | smoothness priors, and the margin grows with respiration | §2 |
| Motion | CHROM/POS hold, GREEN/ICA break — and SNR *rises* as they break | §3 |
| Skin mask | keep it | §4 |
| Spectrum | see the crossing point vs noise | §5 |
| Zero-padding | 8× plus parabolic interpolation, well below the bin spacing | §6 |

**The load-bearing caveat, once more:** every number above comes from synthetic
clips whose ground truth we injected. They justify the *design*. They say
nothing about accuracy on a real face — which is what notebook 02 and
UBFC-rPPG are for, and why the dataset request is still the critical path.
